In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
from utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark =get_spark()
SILVER_PATH = '../data/silver'
GOLD_PATH = '../data/gold' 

df_silver = spark.read.format('delta').load(SILVER_PATH)
print(f'Silver rows loaded: {df_silver.count()}')
df_silver.printSchema()

26/04/07 10:19:17 WARN Utils: Your hostname, Arjuns-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.234 instead (on interface en0)
26/04/07 10:19:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/arjunpillai/.ivy2/cache
The jars for the packages stored in: /Users/arjunpillai/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6ae06c39-1cc8-4f55-b879-a52954472197;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 78ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	-------------------------------------------------------------

:: loading settings :: url = jar:file:/Users/arjunpillai/Desktop/recidivai/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


26/04/07 10:19:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/07 10:19:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/07 10:19:18 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/07 10:19:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Silver rows loaded: 4743
root
 |-- id: integer (nullable = true)
 |-- is_violent_recid: integer (nullable = true)
 |-- v_decile_score: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- sex: string (nullable = true)
 |-- age_cat: string (nullable = true)
 |-- juv_fel_count: integer (nullable = true)
 |-- juv_misd_count: integer (nullable = true)
 |-- priors_count14: integer (nullable = true)
 |-- c_charge_degree: string (nullable = true)



In [2]:
df_gold = (
    df_silver
    .withColumn(
        'is_juvenile_offender',
        F.when(
            (F.col('juv_fel_count') + F.col('juv_misd_count')) > 0, 1
        ).otherwise(0)
    )
    .withColumn(
        'prior_crime_density',
        (F.col('priors_count14') / F.greatest(
            F.least(F.col('age') - F.lit(18), F.lit(14)),
            F.lit(1)
        )).cast(DoubleType())
    )
    .withColumn(
        'high_prior_count',
        F.when(F.col('priors_count14') > 3, 1).otherwise(0)
    )
    .withColumn(
        'charge_severity_score',
        F.when(F.col('c_charge_degree') == 'F', 2)
        .when(F.col('c_charge_degree') == 'M', 1)
        .otherwise(0)
        .cast(IntegerType())
    )
    .withColumn(
        'sex_binary',
        F.when(F.col('sex') == 'Male', 1).otherwise(0).cast(IntegerType())
    )
    .select(
        'id',
        'is_violent_recid',
        'v_decile_score',
        'race',
        'age',
        'priors_count14',
        'juv_fel_count',
        'juv_misd_count',
        'is_juvenile_offender',
        'prior_crime_density',
        'high_prior_count',
        'charge_severity_score',
        'sex_binary'
    )
)

print(f'Gold rows: {df_gold.count()}')
df_gold.printSchema()
df_gold.show(5)


Gold rows: 4743
root
 |-- id: integer (nullable = true)
 |-- is_violent_recid: integer (nullable = true)
 |-- v_decile_score: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- priors_count14: integer (nullable = true)
 |-- juv_fel_count: integer (nullable = true)
 |-- juv_misd_count: integer (nullable = true)
 |-- is_juvenile_offender: integer (nullable = false)
 |-- prior_crime_density: double (nullable = true)
 |-- high_prior_count: integer (nullable = false)
 |-- charge_severity_score: integer (nullable = false)
 |-- sex_binary: integer (nullable = false)

+---+----------------+--------------+----------------+---+--------------+-------------+--------------+--------------------+-------------------+----------------+---------------------+----------+
| id|is_violent_recid|v_decile_score|            race|age|priors_count14|juv_fel_count|juv_misd_count|is_juvenile_offender|prior_crime_density|high_prior_count|charge_severity_score|sex_b

In [3]:
(
    df_gold
    .write
    .format('delta')
    .mode('overwrite')
    .save(GOLD_PATH)
)

print(f'Gold layer written to {GOLD_PATH}')

df_gold.describe(['age', 'priors_count14', 'prior_crime_density',
                  'charge_severity_score']).show()

Gold layer written to ../data/gold
+-------+------------------+------------------+-------------------+---------------------+
|summary|               age|    priors_count14|prior_crime_density|charge_severity_score|
+-------+------------------+------------------+-------------------+---------------------+
|  count|              4743|              4743|               4743|                 4743|
|   mean| 36.00295171832174|2.6772085178157283| 0.2539202902422383|   1.6072106261859582|
| stddev|12.200714544727633|4.0842768290761144| 0.3766541565536131|  0.48842213105798227|
|    min|                18|                 0|                0.0|                    1|
|    max|                83|                38|                4.0|                    2|
+-------+------------------+------------------+-------------------+---------------------+

